# 브랜드 노출 임베딩 분석
**질문 유형 / 시술별 브랜드 노출 패턴 + Latent Space 분석**

| 분석 | 내용 |
|------|------|
| 분석 1 | 질문 유형(추천/비교/후기/전문가) × 브랜드 등장 히트맵 |
| 분석 2 | 시술(procedure) × 브랜드 등장 히트맵 |
| 분석 3 | 브랜드 공동출현 클러스터링 |
| 분석 4 | Q&A 페어 Latent Space (질문유형 / 시술별 색상) |
| 분석 5 | 브랜드 Latent Space (포지셔닝 맵) |

In [ ]:
# Colab 환경에서만 실행
import sys
if 'google.colab' in sys.modules:
    !apt-get install -y -q fonts-nanum
    !pip install -q openai scikit-learn

In [ ]:
import platform
import time
from collections import Counter, defaultdict

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from openai import OpenAI
from sklearn.manifold import TSNE

# 한글 폰트
if platform.system() == 'Darwin':
    matplotlib.rcParams['font.family'] = 'AppleGothic'
else:
    matplotlib.rcParams['font.family'] = 'NanumBarunGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

print('임포트 완료')

In [ ]:
# ── 설정 ──────────────────────────────────────────────────────────────
API_KEY      = ""                   # OpenAI API 키
CSV_PATH     = "dummy_data.csv"     # 데이터 CSV 경로
REGION       = "강남"                # 지역명
INDUSTRY     = "안과"                # 업종
MODEL        = "text-embedding-3-small"
TOP_N_BRANDS = 20                   # 분석할 상위 브랜드 수

# 컬럼명 매핑 (CSV 컬럼명이 다를 경우 수정)
COL = {
    'procedure': 'procedure',
    'attribute': 'attribute',
    'type':      '유형',        # 추천 / 비교 / 후기 / 전문가
    'number':    '번호',
    'brands':    '언급된 브랜드',
}

client = OpenAI(api_key=API_KEY)
print('설정 완료')

In [ ]:
# ── 데이터 로드 및 전처리 ───────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

# 컬럼 표준화
df = df.rename(columns={v: k for k, v in COL.items()})

# 브랜드 파싱 ("A안과, B안과" → ['A안과', 'B안과'])
def parse_brands(s):
    if pd.isna(s) or str(s).strip() == '':
        return []
    return [b.strip() for b in str(s).split(',') if b.strip()]

df['brand_list'] = df['brands'].apply(parse_brands)
df['has_brand']  = df['brand_list'].apply(lambda b: len(b) > 0)

# 질문 텍스트 구성 (유형별 템플릿)
TEMPLATES = {
    '추천':   lambda p, a: f"{REGION}에서 {p}을(를) {a} 측면에서 잘하는 {INDUSTRY} 추천해줘",
    '비교':   lambda p, a: f"{REGION} {INDUSTRY}들의 {p} {a} 비교해줘",
    '후기':   lambda p, a: f"{REGION} {p} {a} 관련 {INDUSTRY} 후기 알려줘",
    '전문가': lambda p, a: f"전문가 추천 {REGION} {p} {a} {INDUSTRY}",
}

def make_question(row):
    tmpl = TEMPLATES.get(row['type'], lambda p, a: f"{p} {a} {row['type']}")
    return tmpl(row['procedure'], row['attribute'])

df['question'] = df.apply(make_question, axis=1)

print(f'총 {len(df):,}행')
print(f'유형: {sorted(df["type"].unique())}')
print(f'procedure: {sorted(df["procedure"].unique())}')
print(f'attribute: {sorted(df["attribute"].unique())}')
print(f'\n브랜드 등장 비율: {df["has_brand"].mean():.1%}')
print(f'\n예시 질문:')
for q in df['question'].drop_duplicates().head(4):
    print(f'  {q}')

In [ ]:
# ── Q&A 페어 집계 (procedure × attribute × 유형 = 100개) ───────────────
# 10번 반복을 하나로 합쳐서 Q&A 페어 텍스트 생성

def agg_group(g):
    all_brands = sorted(set(b for bl in g['brand_list'] for b in bl))
    return pd.Series({
        'question':    g['question'].iloc[0],
        'brand_rate':  g['has_brand'].mean(),
        'all_brands':  all_brands,
        'brand_counts': dict(Counter(b for bl in g['brand_list'] for b in bl)),
        'n_iter':      len(g),
    })

qa = df.groupby(['procedure', 'attribute', 'type', 'question']).apply(agg_group).reset_index()

# Q&A 페어 텍스트: "질문 → 브랜드1, 브랜드2, ..."
qa['brands_text'] = qa['all_brands'].apply(
    lambda bl: ', '.join(bl) if bl else '없음'
)
qa['qa_text'] = qa['question'] + ' → ' + qa['brands_text']

print(f'Q&A 페어: {len(qa)}개  (= procedure × attribute × 유형)')
print(f'\n예시 Q&A 페어:')
for _, row in qa.head(3).iterrows():
    print(f'  [{row["type"]}] {row["qa_text"][:80]}')

In [ ]:
# ── 임베딩 ─────────────────────────────────────────────────────────────
def get_embeddings(texts, batch_size=100):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp  = client.embeddings.create(model=MODEL, input=batch)
        all_vecs.extend([d.embedding for d in resp.data])
        if i + batch_size < len(texts):
            time.sleep(0.3)
    return np.array(all_vecs)

# Q&A 페어 임베딩 (100개)
print(f'[1/2] Q&A 페어 {len(qa)}개 임베딩 중...')
qa_vecs = get_embeddings(qa['qa_text'].tolist())
print(f'      완료: {qa_vecs.shape}')

# 상위 브랜드 임베딩
brand_counter = Counter(b for bl in df['brand_list'] for b in bl)
top_brands    = [b for b, _ in brand_counter.most_common(TOP_N_BRANDS)]
print(f'\n[2/2] 브랜드 {len(top_brands)}개 임베딩 중...')
brand_vecs = get_embeddings(top_brands)
print(f'      완료: {brand_vecs.shape}')

# 재사용을 위해 저장
np.save('qa_vecs.npy', qa_vecs)
np.save('brand_vecs.npy', brand_vecs)
qa.to_csv('qa_agg.csv', index=False, encoding='utf-8-sig')
print('\n임베딩 저장 완료 (qa_vecs.npy, brand_vecs.npy, qa_agg.csv)')

In [ ]:
# ── 저장된 임베딩 불러오기 (API 재호출 없이 재실행 시 사용) ─────────────
# qa_vecs   = np.load('qa_vecs.npy')
# brand_vecs = np.load('brand_vecs.npy')
# qa = pd.read_csv('qa_agg.csv')
# qa['all_brands']  = qa['all_brands'].apply(eval)
# qa['brand_counts'] = qa['brand_counts'].apply(eval)
print('(필요 시 위 주석 해제)')

In [ ]:
# ── 공통 헬퍼 ──────────────────────────────────────────────────────────
def tsne_2d(vecs):
    n    = len(vecs)
    perp = min(max(n // 3, 2), 30)
    return TSNE(n_components=2, perplexity=perp,
                random_state=42, max_iter=1000).fit_transform(vecs)

TYPE_COLORS = {
    '추천':   '#E8455A',
    '비교':   '#4A90D9',
    '후기':   '#5CB85C',
    '전문가': '#F0AD4E',
}
PROC_PALETTE = dict(zip(
    sorted(df['procedure'].unique()),
    sns.color_palette('tab10', df['procedure'].nunique()),
))

print('헬퍼 준비 완료')

## 분석 1 — 질문 유형 × 브랜드 등장 히트맵
어떤 질문 유형(추천/비교/후기/전문가)에서 어떤 브랜드가 잘 나오는가?

In [ ]:
types     = ['추천', '비교', '후기', '전문가']
type_brand = defaultdict(Counter)

for _, row in df.iterrows():
    for brand in row['brand_list']:
        type_brand[row['type']][brand] += 1

mat_type = pd.DataFrame(
    {t: [type_brand[t].get(b, 0) for b in top_brands] for t in types},
    index=top_brands,
)
# 유형별 총 등장 수로 정규화 → 비율
mat_type_norm = mat_type.div(mat_type.sum(axis=0).replace(0, 1), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

sns.heatmap(mat_type, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.4, ax=axes[0], cbar_kws={'label': '등장 횟수'})
axes[0].set_title('질문 유형별 브랜드 등장 횟수', fontsize=13, fontweight='bold')
axes[0].set_xlabel('질문 유형')
axes[0].set_ylabel('브랜드')

sns.heatmap(mat_type_norm, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.4, ax=axes[1], cbar_kws={'label': '비율 (유형 내)'})
axes[1].set_title('질문 유형별 브랜드 등장 비율\n(유형 안에서 이 브랜드가 차지하는 비중)',
                   fontsize=13, fontweight='bold')
axes[1].set_xlabel('질문 유형')

plt.suptitle('분석 1: 어떤 질문 유형에 어떤 브랜드가 잘 나오는가?',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('analysis1_type_brand.png', dpi=150, bbox_inches='tight')
plt.show()

# 유형별 Top 3 브랜드
print('\n유형별 Top 3 브랜드:')
for t in types:
    top3 = mat_type[t].nlargest(3)
    print(f'  [{t}] ' + ' | '.join(f'{b}({c:.0f}회)' for b, c in top3.items()))

## 분석 2 — 시술(procedure) × 브랜드 등장 히트맵
어떤 시술을 물어볼 때 어떤 브랜드가 강한가?

In [ ]:
procedures = sorted(df['procedure'].unique())
proc_brand = defaultdict(Counter)

for _, row in df.iterrows():
    for brand in row['brand_list']:
        proc_brand[row['procedure']][brand] += 1

mat_proc = pd.DataFrame(
    {p: [proc_brand[p].get(b, 0) for b in top_brands] for p in procedures},
    index=top_brands,
)

# 브랜드별 가장 강한 시술 찾기
brand_best_proc = mat_proc.idxmax(axis=1)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(mat_proc, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.4, ax=ax, cbar_kws={'label': '등장 횟수'})
ax.set_title('분석 2: 시술별 브랜드 등장 횟수\n(어떤 시술 질문에 어떤 브랜드가 강한가)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('시술 (procedure)')
ax.set_ylabel('브랜드')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('analysis2_proc_brand.png', dpi=150, bbox_inches='tight')
plt.show()

print('브랜드별 가장 강한 시술:')
for brand, proc in brand_best_proc.items():
    cnt = mat_proc.loc[brand, proc]
    print(f'  {brand:<20} → {proc} ({cnt:.0f}회)')

## 분석 3 — 브랜드 공동출현 클러스터링
어떤 브랜드들이 같은 답변에 함께 자주 등장하는가?

In [ ]:
brand_to_idx = {b: i for i, b in enumerate(top_brands)}
cooc = np.zeros((len(top_brands), len(top_brands)))

for _, row in df.iterrows():
    brands_in_row = [b for b in row['brand_list'] if b in brand_to_idx]
    for i, b1 in enumerate(brands_in_row):
        for b2 in brands_in_row[i + 1:]:
            cooc[brand_to_idx[b1], brand_to_idx[b2]] += 1
            cooc[brand_to_idx[b2], brand_to_idx[b1]] += 1

cooc_df = pd.DataFrame(cooc, index=top_brands, columns=top_brands)

# 계층적 클러스터링 히트맵
g = sns.clustermap(
    cooc_df,
    cmap='YlOrRd',
    figsize=(13, 11),
    annot=True, fmt='.0f',
    linewidths=0.3,
    method='ward',
    cbar_kws={'label': '공동 등장 횟수'},
)
g.ax_heatmap.set_title(
    '분석 3: 브랜드 공동출현 클러스터링\n(함께 자주 등장하는 브랜드 그룹)',
    fontsize=13, fontweight='bold', pad=20,
)
plt.savefig('analysis3_brand_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 공동출현 쌍
print('가장 자주 함께 등장하는 브랜드 쌍 Top 10:')
pairs = []
for i, b1 in enumerate(top_brands):
    for j, b2 in enumerate(top_brands):
        if j > i:
            pairs.append((b1, b2, cooc[i, j]))
for b1, b2, cnt in sorted(pairs, key=lambda x: -x[2])[:10]:
    print(f'  {b1} + {b2}: {cnt:.0f}회')

## 분석 4 — Q&A 페어 Latent Space (t-SNE)
임베딩 공간에서 질문들이 어떻게 분포하는가? (질문 유형별 / 시술별)

In [ ]:
qa_2d = tsne_2d(qa_vecs)

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# ── 차트 1: 질문 유형별 색상 ──────────────────────────────────────────
ax = axes[0]
for utype, color in TYPE_COLORS.items():
    mask = qa['type'] == utype
    ax.scatter(
        qa_2d[mask, 0], qa_2d[mask, 1],
        c=color, s=80, alpha=0.8, edgecolors='grey', lw=0.4,
        label=utype,
    )
ax.legend(title='질문 유형', fontsize=9)
ax.set_title('Q&A Latent Space\n질문 유형별', fontsize=12, fontweight='bold')
ax.set_xlabel('t-SNE Dim 1')
ax.set_ylabel('t-SNE Dim 2')

# ── 차트 2: 시술별 색상 ───────────────────────────────────────────────
ax = axes[1]
for proc, color in PROC_PALETTE.items():
    mask = qa['procedure'] == proc
    ax.scatter(
        qa_2d[mask, 0], qa_2d[mask, 1],
        c=[color], s=80, alpha=0.8, edgecolors='grey', lw=0.4,
        label=proc,
    )
ax.legend(title='시술', fontsize=9)
ax.set_title('Q&A Latent Space\n시술별', fontsize=12, fontweight='bold')
ax.set_xlabel('t-SNE Dim 1')

# ── 차트 3: 브랜드 노출률(brand_rate)로 색상 ─────────────────────────
ax = axes[2]
sc = ax.scatter(
    qa_2d[:, 0], qa_2d[:, 1],
    c=qa['brand_rate'], cmap='RdYlGn',
    s=100, vmin=0, vmax=1,
    alpha=0.85, edgecolors='grey', lw=0.4,
)
plt.colorbar(sc, ax=ax, label='Brand Rate (노출률)')
ax.set_title('Q&A Latent Space\n브랜드 노출률', fontsize=12, fontweight='bold')
ax.set_xlabel('t-SNE Dim 1')

# 노출률 높은 상위 5개 라벨 표시
top5_idx = qa['brand_rate'].nlargest(5).index
for idx in top5_idx:
    row = qa.loc[idx]
    ax.annotate(
        f"{row['procedure']}\n{row['type']}",
        (qa_2d[idx, 0], qa_2d[idx, 1]),
        fontsize=7, ha='center', va='bottom',
        xytext=(0, 6), textcoords='offset points',
    )

plt.suptitle('분석 4: Q&A 페어 Latent Space', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('analysis4_qa_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 분석 5 — 브랜드 Latent Space (포지셔닝 맵)
브랜드들이 임베딩 공간에서 어떻게 배치되는가?

In [ ]:
brand_2d = tsne_2d(brand_vecs)

# 브랜드별 주 등장 시술 (가장 많이 나온 시술)
brand_best_proc_color = [
    PROC_PALETTE[mat_proc.loc[b].idxmax()] if b in mat_proc.index else '#aaa'
    for b in top_brands
]
# 브랜드별 주 등장 유형
brand_best_type_color = [
    TYPE_COLORS.get(mat_type.loc[b].idxmax(), '#aaa') if b in mat_type.index else '#aaa'
    for b in top_brands
]
brand_sizes = [80 + brand_counter[b] * 4 for b in top_brands]

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

for ax, colors, palette, palette_title in [
    (axes[0], brand_best_proc_color, PROC_PALETTE, '주 등장 시술'),
    (axes[1], brand_best_type_color, TYPE_COLORS,  '주 등장 유형'),
]:
    ax.scatter(
        brand_2d[:, 0], brand_2d[:, 1],
        c=colors, s=brand_sizes,
        alpha=0.85, edgecolors='grey', lw=0.5,
    )
    for i, brand in enumerate(top_brands):
        ax.annotate(
            brand,
            (brand_2d[i, 0], brand_2d[i, 1]),
            fontsize=8, ha='center', va='bottom',
            xytext=(0, 5), textcoords='offset points',
        )
    for label, color in palette.items():
        ax.scatter([], [], c=[color] if isinstance(color, str) else [color],
                   label=label, s=80)
    ax.legend(title=palette_title, fontsize=9, loc='upper right')
    ax.set_xlabel('t-SNE Dim 1')
    ax.set_ylabel('t-SNE Dim 2')

axes[0].set_title('브랜드 포지셔닝 맵\n색상: 주 등장 시술 / 크기: 총 언급 빈도',
                   fontsize=12, fontweight='bold')
axes[1].set_title('브랜드 포지셔닝 맵\n색상: 주 등장 질문 유형 / 크기: 총 언급 빈도',
                   fontsize=12, fontweight='bold')

plt.suptitle('분석 5: 브랜드 Latent Space — 포지셔닝 맵',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('analysis5_brand_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약 출력

In [ ]:
print('=' * 60)
print('분석 요약')
print('=' * 60)

print(f'\n▶ 총 데이터: {len(df):,}행  |  고유 Q&A 페어: {len(qa)}개')
print(f'▶ 전체 브랜드 등장 비율: {df["has_brand"].mean():.1%}')

print('\n▶ 질문 유형별 브랜드 등장률:')
for t in types:
    rate = df[df['type'] == t]['has_brand'].mean()
    print(f'   {t:<6}: {rate:.1%}')

print('\n▶ 시술별 브랜드 등장률:')
for p in procedures:
    rate = df[df['procedure'] == p]['has_brand'].mean()
    print(f'   {p:<12}: {rate:.1%}')

print(f'\n▶ 가장 많이 언급된 브랜드 Top 5:')
for brand, cnt in brand_counter.most_common(5):
    print(f'   {brand:<20}: {cnt}회')

print('\n▶ 저장된 파일:')
for f in ['analysis1_type_brand.png', 'analysis2_proc_brand.png',
          'analysis3_brand_cooccurrence.png', 'analysis4_qa_tsne.png',
          'analysis5_brand_tsne.png']:
    print(f'   {f}')